# Stage 1: Select Position Openers

Select wallets whose opening BUYs are worth copying.
Uses volatility-based wallet metrics + threshold scoring (matching reference notebook).

Grid-search over selection thresholds to maximize **copyable PnL from opening buys** on the validation split.

**Output:** `stage1_result.json` with best selection params.

In [17]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    evaluate_wallet_group,
    evaluate_wallet_group_openers,
    select_copyable_group,
    run_grid_search,
    save_stage_result,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [18]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

Markets: 1974837
Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...
Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00
Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z
Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)
  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)
  Total: 14,250,603 trades  (56,875 markets)


In [19]:
df_test['end_date_iso'].min()

'2026-06-24T00:00:00Z'

## Compute wallet metrics on training data

In [20]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "opening_roi", "opening_pnl", "opening_copyable_pnl", "copyable_pnl", "num_buckets"]].head(10)

Wallets with metrics: 3220


,wallet,buy_roi,opening_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0088,0.0062,9.4227,-0.1185,-0.5270,38
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0394,0.0681,52.7871,-3.1619,2.5901,208
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,-0.0700,-1.0000,-1.0000,0.0000,0.0000,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.0288,0.6033,59.3074,60.0349,57.7247,34
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,-0.0094,-0.0877,-17.1649,-20.6348,-17.5761,299
5,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0084,0.0194,5.7705,1.4397,3.2916,23
6,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0099,0.0102,27.6810,-7.3452,-0.6680,844
7,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.0726,0.1260,899.2771,95.0644,-401.0333,989
8,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.0001,-0.0042,-42.1920,-25.7250,30.6519,1174
9,0x01f2a8baabe17c2541d1e3091220991f257ac3de,0.0051,0.0036,179.5891,-761.1021,-3381.9584,39804


## Baseline selection (reference defaults)

In [32]:
copyable_group = select_copyable_group(
    wallet_vol,
    min_buy_roi=0.03,
    min_num_buckets=20,
    min_num_markets=15,
    max_drawdown_to_pnl=0.20,
    max_top_market_pnl_pct=0.25,
    max_market_pnl_hhi=0.30,
    min_total_notional=5_000,
    min_opening_roi=0.0,
    min_opening_pnl=0,
    min_opening_copyable_roi=0.0,
)
print(f"Copyable group: {len(copyable_group)} wallets")
show_cols = ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
             "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
copyable_group[[c for c in show_cols if c in copyable_group.columns]].head(15)

Copyable group: 34 wallets


,wallet,opening_roi,opening_copyable_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,buy_roi,num_buckets
0,0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3,0.7477,0.7923,581.4098,175.8440,35.0337,0.0983,404
1,0x35aff83368c69c47af04ee2d99330154f22f1ca6,0.6031,0.5008,522.7273,207.6186,205.3476,0.1237,489
2,0x919698b19427cbe6945b0dc823f2d9e126a4d934,0.0948,0.1978,580.1146,203.5358,442.0758,0.0719,785
3,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,0.1696,0.1925,1764.4774,596.3652,2829.5819,0.0558,1459
4,0x07d601375c9bbb9037ad3c7a8f8fa0deff8164fb,0.2075,0.1695,425.9361,61.4549,55.5609,0.0839,353
5,0x4b371de80c78f6484771ff72a961b63566192e48,0.1169,0.1690,347.6881,270.0447,504.7343,0.0819,344
6,0x7231a52f9de4fda5218d0e63f30a3499a4535afe,0.2210,0.1655,526.5890,123.0785,967.6824,0.0614,625
7,0x0e09d1f32963451855e429c384be6499dd0e5eef,0.1982,0.1438,404.3683,162.0869,146.1969,0.1372,98
8,0x45e606f7849330adad37875f835fbdd1b7868fca,0.0688,0.1365,543.3619,318.2269,451.4776,0.0618,983
9,0x46532d38063a22045404aeead6a9f3c49e75fc2b,0.0644,0.1318,264.6895,145.4317,234.4447,0.0533,136


## Baseline evaluation (reference format)

In [33]:
wallet_set = set(copyable_group["wallet"])

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


*** TRAIN copyable group ***
  Open  : wallet_pnl=  27747.79  roi=0.0678  |  copyable_pnl=   5221.69  roi=0.0767
  Total : wallet_pnl=  74947.61  roi=0.0786  |  copyable_pnl=  20648.58  roi=0.0900

*** VAL copyable group ***
  Open  : wallet_pnl=  28307.64  roi=0.0604  |  copyable_pnl=   2130.09  roi=0.0243
  Total : wallet_pnl=  51795.45  roi=0.0458  |  copyable_pnl=   4116.80  roi=0.0153

*** TEST copyable group ***
  Open  : wallet_pnl=  35423.22  roi=0.0857  |  copyable_pnl=   1735.37  roi=0.0270
  Total : wallet_pnl=  50356.89  roi=0.0640  |  copyable_pnl=    219.47  roi=0.0014


## Grid search

Vary selection thresholds to maximize copyable PnL from opening buys on the validation split.

In [42]:
param_grid = dict(
    min_buy_roi=[0.03, 0.05, 0.07],
    min_num_buckets=[15],
    min_num_markets=[10, 20],
    max_drawdown_to_pnl=[0.3],
    max_top_market_pnl_pct=[1],
    max_market_pnl_hhi=[0.30],
    min_total_notional=[1_000],
    min_opening_roi=[0.05, 0.07],
    min_opening_pnl=[200],
    min_opening_copyable_roi=[0.05, 0.07],
)

print(f"Grid: {np.prod([len(v) for v in param_grid.values()]):.0f} combos")

Grid: 24 combos


In [43]:
res_df = run_grid_search(param_grid, wallet_vol, df_val)
# print(f'open_copyable_pnl: {res_df["open_copyable_pnl"].max():.0f}, open_copyable_roi: {res_df["open_copyable_pnl"].max():.4f}')
res_df.head()

best_row = res_df.iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}
best_group = select_copyable_group(wallet_vol, **best_params)

print(f"Best config (val open copyable_pnl={best_row['open_copyable_pnl']:.2f}):")
print(best_params)
print(f"  wallets: {best_row['wallets']:.0f}  open_wallets: {best_row['open_wallets']:.0f}")

Grid: 24 combos, 8 workers
  [24/24] 3.2s elapsed
Done: 24 configs in 3.2s
Best config (val open copyable_pnl=2554.68):
{'min_buy_roi': np.float64(0.03), 'min_num_buckets': np.float64(15.0), 'min_num_markets': np.float64(10.0), 'max_drawdown_to_pnl': np.float64(0.3), 'max_top_market_pnl_pct': np.float64(1.0), 'max_market_pnl_hhi': np.float64(0.3), 'min_total_notional': np.float64(1000.0), 'min_opening_roi': np.float64(0.07), 'min_opening_pnl': np.float64(200.0), 'min_opening_copyable_roi': np.float64(0.05)}
  wallets: 44  open_wallets: 33


In [44]:
print('top 10 results')
res_df.head(10)

top 10 results


,min_buy_roi,min_num_buckets,min_num_markets,max_drawdown_to_pnl,max_top_market_pnl_pct,max_market_pnl_hhi,min_total_notional,min_opening_roi,min_opening_pnl,min_opening_copyable_roi,open_copyable_pnl,open_wallet_pnl,open_wallets,total_copyable_pnl,wallets,elapsed
1,0.0300,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,2554.6842,36142.2915,33,-1022.6342,44,1.2817
0,0.0300,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,2521.4657,35591.0352,31,-1087.4019,41,1.2703
5,0.0300,15,10,0.3000,1,0.3000,1000,0.0500,200,0.0500,2411.3368,41909.8643,38,-815.5219,49,1.5001
6,0.0300,15,20,0.3000,1,0.3000,1000,0.0500,200,0.0500,2378.1183,41358.6080,36,-880.2896,46,1.5965
9,0.0500,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,2323.2836,30998.1147,28,696.4312,39,1.0072
14,0.0500,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,2290.0652,30446.8585,26,631.6635,36,0.8766
8,0.0500,15,10,0.3000,1,0.3000,1000,0.0500,200,0.0500,1983.8621,31049.0864,30,549.3899,41,1.0102
10,0.0500,15,20,0.3000,1,0.3000,1000,0.0500,200,0.0500,1950.6436,30497.8301,28,484.6222,38,1.0038
4,0.0300,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0700,1758.5508,33666.7217,28,-1004.7628,39,1.2994
7,0.0300,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0700,1725.3324,33115.4654,26,-1069.5305,36,1.4838


## Stage 1 results

In [45]:
if best_group is not None and not best_group.empty:
    print(f"Copyable group: {len(best_group)} wallets")
    cols = [c for c in ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
                         "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
            if c in best_group.columns]
    print(best_group[cols].head(15).to_string())

Copyable group: 44 wallets
                                        wallet  opening_roi  opening_copyable_roi  opening_pnl  opening_copyable_pnl  copyable_pnl  buy_roi  num_buckets
0   0x0a59a7f2a870392a5f555aaa11f49d612e748e5c       0.9370                1.1262    1094.7708              482.7124       78.2269   0.0373         1106
1   0x76305b2a31e7ec35189650c93e8df2a15b92789d       0.6783                0.9857     478.4984              313.4714      327.5507   0.2266          351
2   0xf1e18ec32b2f1e123bc098e3956e6fd00012c152       0.4716                0.8065     744.3432              393.6461     1239.2025   0.1159          452
3   0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3       0.7477                0.7923     581.4098              175.8440       35.0337   0.0983          404
4   0x515ca60f91f49e5e4aa23ab60a6edec3c3917587       0.2649                0.6472     441.9531              191.4272       24.9526   0.0647          189
5   0x9c68b13a2c9b6d2e80826f26cea746cc22ba7936       0.

In [46]:
wallet_set = set(best_group["wallet"])
print(f"\nSelected {len(wallet_set)} wallets")
for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


Selected 44 wallets

*** TRAIN copyable group ***
  Open  : wallet_pnl=  34061.61  roi=0.2179  |  copyable_pnl=   7466.21  roi=0.1945
  Total : wallet_pnl= 105813.65  roi=0.2254  |  copyable_pnl=  26290.48  roi=0.2057

*** VAL copyable group ***
  Open  : wallet_pnl=  36142.29  roi=0.1208  |  copyable_pnl=   2554.68  roi=0.0436
  Total : wallet_pnl=  82083.06  roi=0.0864  |  copyable_pnl=   8323.24  roi=0.0435

*** TEST copyable group ***
  Open  : wallet_pnl=  33179.11  roi=0.1165  |  copyable_pnl=   4026.98  roi=0.0817
  Total : wallet_pnl=  73027.52  roi=0.0696  |  copyable_pnl=   6519.47  roi=0.0338


## Save stage 1 result

In [47]:
import json
from datetime import datetime, timezone
from pathlib import Path

wallet_cols = [
    "wallet", "buy_roi", "opening_roi", "opening_pnl",
    "opening_copyable_roi", "opening_copyable_pnl",
    "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "max_drawdown_to_pnl", "top_market_pnl_pct", "market_pnl_hhi",
    "wallet_quality",
]
wallet_records = best_group[[c for c in wallet_cols if c in best_group.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

metadata = {
    "type": "openers",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": len(best_group),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_open_copyable_pnl": float(best_row["open_copyable_pnl"]),
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = Path("stage1_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")

Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_result.json


In [48]:
df = df_test[
    (df_test["wallet"].isin(wallet_set))
    & (df_test["side"] == "BUY") & (df_test["position"] == df_test["quantity"])
    ]
len(df)

17683

In [49]:
df['copyable_pnl'].sum()

np.float64(4026.9775633175936)